这一节之前，模型已经完成了 SFT。跑出来的模型确实会遵循指令、会用助手语气说话、遇到危险请求也知道拒绝。但很快就发现另一个更尖锐的问题：当两个回答都"像人话"时，模型怎么学会更偏向人类更喜欢的那一个？

SFT 解决的是"会不会说话"的问题，而 RLHF 和 DPO 要解决的是"说得好不好"的问题。

再提一嘴，前面的 GRPO 是"可验证任务上的在线策略优化"，这里的 DPO 则是"离线偏好数据上的直接对比对齐"。两者都在做 alignment，但信号来源完全不同。GRPO 吃的是 prompt -> rollout -> reward；DPO 吃的是 (prompt, chosen, rejected) 这种偏好对。DPO 原论文的核心结论是：在标准 RLHF 的"奖励最大化 + KL 约束"目标下，最优策略可以写成参考策略与奖励函数的闭式关系，于是可以把偏好学习直接改写成一个分类损失，而不再需要显式奖励模型和 PPO 采样环。
## 8.1 RLHF
经典 RLHF（以 PPO 路线为代表）的完整流程大致是：先收集人类偏好比较数据，然后训练一个奖励模型去拟合"人更喜欢哪个回答"，再让 policy 模型反复采样生成回答，用奖励模型给这些回答打分，最后用 PPO 更新 policy 参数，同时靠 reference 模型施加 KL 约束防止策略漂得太远。

InstructGPT 是这条路线的代表作：SFT → Reward Model → PPO，三步走。能工作，但训练链条长、资源消耗高，而且每一段都有自己的不稳定性来源。

PPO 阶段需要同时维护 actor 模型、critic 模型、reference 模型和 reward 模型，四个模型在显存里同时运转，训练调度和超参调试的复杂度都不低。

DPO 没有否定 RLHF 的目标，而是对同一目标换了一种解法。DPO 的核心主张是：在带 KL 正则的最优策略表达式下，奖励可以被重参数化成 **当前策略相对参考策略的对数比值**，于是 `(chosen, rejected)` 偏好对就能直接变成一个 logistic 分类损失，不再需要显式奖励模型和 PPO 采样环。
> Your Language Model is Secretly a Reward Model。

## 8.2 DPO 损失函数
DPO 把 PPO 时期庞大的四驾马车精简成了一个基于概率差值对比的双模型结构：

- **在线策略（policy, $\pi_\theta$）**：当前正在被训练、处于学习状态的目标模型。
- **冻结参考（reference, $\pi_{\text{ref}}$）**：作为基准的原始 SFT 模型，参数完全冻结，用于锚定模型的底层语言能力，防止策略在偏好学习中发生灾难性遗忘。

整个训练过程的核心是比较：在给定同一个 prompt $x$ 的前提下，当前策略对 chosen 回答 $y_w$ 的偏好程度，相比于它对 rejected 回答 $y_l$ 的偏好程度，是否比参考模型拉开了更大的差距。

标准 DPO 目标可以写成：

$$
\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = -\mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}}\left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \right]
$$
> 这里 `w/l` 表示 `winner/loser`
> - **$r(x,y)$**：潜在的奖励函数，衡量回答 $y$ 对提示 $x$ 有多“好”。
> - **$\beta$**：控制 KL 惩罚强度的超参数，$\beta>0$。
> - **$\sigma(\cdot)$**：sigmoid 函数，$\sigma(z)=1/(1+e^{-z})$。
> - **$Z(x)$**：配分函数，用于归一化概率分布，只依赖于 $x$。

### 8.2.1 物理含义
这个公式可以从三层结构来理解它的物理含义。

第一层，$$\beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$ 可以看成一种**隐式奖励**：当前模型相对于参考模型，到底有多"偏爱"这个回答。当前模型给这个回答的概率比值越大，隐式奖励就越高。

第二层，chosen 和 rejected 的隐式奖励差值 $$\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}$$ 表示："我对好回答的偏爱，是否明显大于我对坏回答的偏爱。"如果这个差值为正且足够大，说明模型确实在偏好方向上做出了正确的区分。

第三层，外层的 `log sigmoid`，本质上是一个二元分类损失。当 chosen 和 rejected 的隐式奖励差距为正且大时，sigmoid 输出接近 1，log 接近 0，loss 小；如果差距为负（即模型更偏好 rejected），sigmoid 输出接近 0，log 为一个绝对值很大的负数，loss 急剧增大。整个优化就是把"chosen 应该胜过 rejected"这件事当成监督信号去拟合。

### 8.2.2 数学准备
#### Bradley-Terry
人类的相对偏好可以用 Bradley-Terry 模型来量化。假设每个回答 $y$ 对应一个潜在奖励值 $r(x,y)$，那么在给定提示 $x$ 下，人类选择 $y_w$ 而非 $y_l$ 的概率为：

$$
P(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big) = \frac{1}{1 + \exp\big(-\big(r(x,y_w) - r(x,y_l)\big)\big)}
$$

这个式子将“偏好比较”转化成了奖励差值经过 sigmoid 映射的概率。经典 RLHF 会用一个独立的奖励模型去拟合 $r(x,y)$，而 DPO 的想法是直接用策略模型来“表达”这个奖励。
#### KL 散度
然后复用一下 KL 散度的约束：
$$
D_{\text{KL}}\big[\pi(y|x) \parallel \pi_{\text{ref}}(y|x)\big] = \mathbb{E}_{y \sim \pi(y|x)}\left[ \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} \right]
$$
在 RLHF 目标中加入 KL 惩罚有两个关键作用：
- 一是防止 reward hacking（模型为了拿高奖励而输出怪异文本）；
- 二是保留基座能力（不让模型忘记预训练和 SFT 阶段学到的语言知识）。
#### RLHF 全局优化目标
标准的 RLHF 优化目标可以形式化地写为：在给定提示分布下，最大化生成回答的奖励期望，并减去一个由 $\beta$ 加权的 KL 散度项：

$$
\max_{\pi} \ \mathbb{E}_{x \sim \mathcal{D},\ y \sim \pi(y|x)} \big[ r(x,y) \big] - \beta \, D_{\text{KL}}\big[ \pi(y|x) \parallel \pi_{\text{ref}}(y|x) \big]
$$
其中 $\beta$ 越大，策略允许偏离参考模型的程度越小。为了推导方便，将最大化转为最小化（取相反数），并将 KL 散度展开：

$$
\min_{\pi} \ \mathbb{E}_{x,\ y \sim \pi} \left[ \beta \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} - r(x,y) \right]
$$
提取公因子 $\beta$，得到：

$$
\min_{\pi} \ \beta \, \mathbb{E}_{x,\ y \sim \pi} \left[ \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} - \frac{1}{\beta}r(x,y) \right]
$$
#### 构造归一化因子并找出最优解
将 $-\frac{1}{\beta}r(x,y)$ 改写为 $-\log\big(\exp(\frac{1}{\beta}r(x,y))\big)$，并合并到对数中：

$$
\min_{\pi} \ \beta \, \mathbb{E}_{x,\ y \sim \pi} \left[ \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x) \exp\!\big(\frac{1}{\beta}r(x,y)\big)} \right]
$$
此时分母 $\pi_{\text{ref}}(y|x) \exp\!\big(\frac{1}{\beta}r(x,y)\big)$ 并不是一个合法的概率分布。为此，引入配分函数 $Z(x)$ 进行归一化：

$$
Z(x) = \sum_{y} \pi_{\text{ref}}(y|x) \exp\!\Big(\frac{1}{\beta}r(x,y)\Big)
$$
于是可以定义理论上的最优分布 $\pi^*(y|x)$：

$$
\pi^*(y|x) = \frac{\pi_{\text{ref}}(y|x) \exp\!\big(\frac{1}{\beta}r(x,y)\big)}{Z(x)}
$$
将 $\pi^*$ 代回优化式，分离出与 $\pi$ 无关的项：

$$
\begin{aligned}
\min_{\pi} &\ \mathbb{E}_{x,\ y \sim \pi} \left[ \log\frac{\pi(y|x)}{\pi^*(y|x)} - \log Z(x) \right] \\
= &\ D_{\text{KL}}\big[ \pi(y|x) \parallel \pi^*(y|x) \big] - \mathbb{E}_x [\log Z(x)]
\end{aligned}
$$

因为 KL 散度恒非负，且 $\log Z(x)$ 与 $\pi$ 无关，所以当且仅当 $\pi(y|x) = \pi^*(y|x)$ 时，上式取得最小值。也就是说，RLHF 目标在 KL 约束下存在唯一的闭式最优解 $\pi^*$。
#### 奖励重参数化
从最优解表达式中解出奖励函数 $r(x,y)$。对等式 $\pi(y|x) = \pi^*(y|x)$ 两边取对数：
$$
\log \pi(y|x) = \log \pi_{\text{ref}}(y|x) + \frac{1}{\beta}r(x,y) - \log Z(x)
$$
移项，得到**隐式奖励**的表达式：

$$
r(x,y) = \beta \log\frac{\pi(y|x)}{\pi_{\text{ref}}(y|x)} + \beta \log Z(x)
$$
这个公式的含义非常深刻：奖励不再是需要单独建模的外部函数，而是由策略模型和参考模型的对数概率差直接给出。换句话说，语言模型本身就暗含了奖励函数的角色。

#### 代入 Bradley-Terry 模型，消除配分函数
现在计算 chosen \(y_w\) 和 rejected \(y_l\) 之间的奖励差值：

$$
\begin{aligned}
r(x,y_w) - r(x,y_l) &=
\left( \beta \log\frac{\pi(y_w|x)}{\pi_{\text{ref}}(y_w|x)} + \beta \log Z(x) \right)
- \left( \beta \log\frac{\pi(y_l|x)}{\pi_{\text{ref}}(y_l|x)} + \beta \log Z(x) \right) \\
&= \beta \log\frac{\pi(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log\frac{\pi(y_l|x)}{\pi_{\text{ref}}(y_l|x)}
\end{aligned}
$$

由于 \(Z(x)\) 只依赖于 \(x\) 而不依赖于具体的 \(y\)，在做减法时完美抵消。这一消除使得我们完全避开了难以计算的归一化项。

将此差值代入 Bradley-Terry 的概率公式中，得到偏好概率：
$$
P(y_w \succ y_l \mid x) = \sigma\!\left( \beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right)
$$
为了用最大似然估计来优化策略参数 \(\theta\)，对每一对偏好数据 \((x, y_w, y_l)\) 取负对数似然，得到 DPO 的最终损失函数：

$$
\mathcal{L}_{\text{DPO}}(\pi_\theta; \pi_{\text{ref}}) = - \mathbb{E}_{(x, y_w, y_l) \sim \mathcal{D}} \Bigg[ \log \sigma\!\left( \beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)} \right) \Bigg]
$$

这就是 DPO 的核心：它将一个带有 KL 约束的奖励最大化问题，转化为了一个简洁的二元分类问题。优化这个损失等价于让模型在 chosen 回答上相对于参考模型提升概率，同时在 rejected 回答上相对于参考模型降低概率，且幅度受 \(\beta\) 调节。

## 8.3 DPO vs GRPO
DPO 和 GRPO 虽然都在做对齐，但定位完全不同。GRPO 需要在线采样、需要 reward/verifier、依赖 old policy 做近端约束，适合数学、代码这类有明确可验证信号的任务。DPO 不需要在线采样和显式奖励模型，只依赖离线的静态偏好对，适合通用助手的偏好精调。
> **GRPO 是“在线采样 + 奖励反馈 + 近端策略更新”，DPO 是“离线偏好对 + 直接对比优化”。**

| 特性 | GRPO | DPO |
|------|------|-----|
| 在线采样 (Rollout) | 需要 | 不需要 |
| 显式奖励模型/验证器 (Reward/Verifier) | 需要 | 不需要 |
| 旧策略 (Old Policy) | 需要 | 不需要 |
| 数据类型 | 在线/准在线数据 | 离线偏好数据 |
| 最佳适用场景 | 数学、代码等有可验证结果的任务 | 通用助手的偏好对齐 |

有研究发现，GRPO 在更大模型上的 reasoning 表现更好，而 DPO 在偏好对齐上更稳定。在工程实践中，一个常见的组合是先 SFT → DPO 做离线的低成本偏好对齐，再针对数学/代码等特定能力叠加 GRPO 强化推理。

In [ ]:
import os
import gzip
import json
import random
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import wandb

from transformers import AutoModelForCausalLM, AutoTokenizer

## 8.4 实验代码

In [ ]:
CFG = {
    # 数据
    "hh_path": "data/hh-rlhf",

    # 模型
    "policy_model_path": "result/sft_qwen_3b_ultraChat_SafetyLlama/checkpoint-6300",
    "reference_model_path": "result/sft_qwen_3b_ultraChat_SafetyLlama/checkpoint-6300",
    # 如果你要严格复现原始脚本，可改成:
    # "reference_model_path": "model/Qwen2.5-3B"

    # 输出
    "output_dir": "result/dpo_qwen_3b_sft",

    # 训练超参数
    "max_length": 512,
    "train_batch_size": 64,             # 逻辑 batch
    "gradient_accumulation_steps": 16,
    "lr": 1e-6,
    "epochs": 1,
    "beta": 0.1,

    # 监控与保存
    "eval_interval": 20,
    "save_interval": 500,
    "wandb_project": "cs336-dpo",
    "wandb_run_name": "qwen3b-dpo-beta0.1-lr1e-6",

    # 设备
    "policy_device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "reference_device": "cuda:0" if torch.cuda.is_available() else "cpu",

    # 其他
    "seed": 42,
}
CFG["micro_batch_size"] = CFG["train_batch_size"] // CFG["gradient_accumulation_steps"]

In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

### 8.4.1 加载 HH-RLHF 数据

In [ ]:
def load_hh_dataset(hh_path: str, seed: int = 42, train_ratio: float = 0.95):
    filenames = [
        "harmless-base/train.jsonl.gz",
        "helpful-base/train.jsonl.gz",
        "helpful-online/train.jsonl.gz",
        "helpful-rejection-sampled/train.jsonl.gz",
    ]

    all_examples = []

    for filename in filenames:
        file_path = os.path.join(hh_path, filename)
        if not os.path.exists(file_path):
            continue

        with gzip.open(file_path, "rt", encoding="utf-8") as f:
            for line in f:
                data = json.loads(line)

                chosen = data.get("chosen", "")
                rejected = data.get("rejected", "")

                c_msg = [m for m in chosen.split("\n\n") if m.strip()]
                r_msg = [m for m in rejected.split("\n\n") if m.strip()]

                if len(c_msg) == 2 and len(r_msg) == 2 and c_msg[0] == r_msg[0]:
                    all_examples.append({
                        "instruction": c_msg[0],
                        "chosen": c_msg[1],
                        "rejected": r_msg[1],
                    })

    rng = random.Random(seed)
    rng.shuffle(all_examples)

    split = int(len(all_examples) * train_ratio)
    train_examples = all_examples[:split]
    val_examples = all_examples[split:]

    return train_examples, val_examples

先抽一条样本看格式：

In [ ]:
train_examples, val_examples = load_hh_dataset(CFG["hh_path"], seed=CFG["seed"])

print("train size =", len(train_examples))
print("val size   =", len(val_examples))

ex = train_examples[0]
print("INSTRUCTION:\n", ex["instruction"][:400])
print("\nCHOSEN:\n", ex["chosen"][:400])
print("\nREJECTED:\n", ex["rejected"][:400])

### 8.4.2 Preference Dataset
1. chosen / rejected 分别编码”
2. “左填充”
3. “尽量保证 response 完整保留下来

In [ ]:
class HHPreferenceDataset(Dataset):
    def __init__(self, examples, tokenizer, max_length: int = 512):
        self.examples = examples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.examples)

    def _process_tokens(self, prompt_tokens, response_tokens):
        """
        拼接 -> 左截断 -> 左填充
        返回:
          input_ids, attention_mask, prompt_end_index
        """
        full_tokens = (prompt_tokens + response_tokens)[-self.max_length:]

        # 在截断后的序列中，prompt 实际保留下来的长度
        actual_prompt_len = max(0, len(full_tokens) - len(response_tokens))

        pad_len = self.max_length - len(full_tokens)
        input_ids = [self.tokenizer.pad_token_id] * pad_len + full_tokens
        attention_mask = [0] * pad_len + [1] * len(full_tokens)

        prompt_end_index = pad_len + actual_prompt_len

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(attention_mask, dtype=torch.long),
            torch.tensor(prompt_end_index, dtype=torch.long),
        )

    def __getitem__(self, idx):
        ex = self.examples[idx]

        template = (
            "Below is an instruction that describes a task.\n\n"
            "### Instruction:\n{}\n\n"
            "### Response:\n"
        )
        prompt = template.format(ex["instruction"])

        prompt_tokens = self.tokenizer.encode(prompt, add_special_tokens=False)
        chosen_tokens = self.tokenizer.encode(ex["chosen"], add_special_tokens=False) + [self.tokenizer.eos_token_id]
        rejected_tokens = self.tokenizer.encode(ex["rejected"], add_special_tokens=False) + [self.tokenizer.eos_token_id]

        c_ids, c_mask, c_p_len = self._process_tokens(prompt_tokens, chosen_tokens)
        r_ids, r_mask, r_p_len = self._process_tokens(prompt_tokens, rejected_tokens)

        return {
            "chosen_input_ids": c_ids,
            "chosen_attention_mask": c_mask,
            "chosen_prompt_len": c_p_len,
            "rejected_input_ids": r_ids,
            "rejected_attention_mask": r_mask,
            "rejected_prompt_len": r_p_len,
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["policy_model_path"])
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_ds_preview = HHPreferenceDataset(
    train_examples,
    tokenizer,
    max_length=CFG["max_length"],
)

sample = train_ds_preview[0]
for k, v in sample.items():
    print(k, v.shape if hasattr(v, "shape") else v)

In [ ]:
# 检查 chosen / rejected decode 是否合理
sample = train_ds_preview[0]

chosen_text = tokenizer.decode(sample["chosen_input_ids"], skip_special_tokens=False)
rejected_text = tokenizer.decode(sample["rejected_input_ids"], skip_special_tokens=False)

print("CHOSEN DECODE:\n", chosen_text[:1200])
print("\nREJECTED DECODE:\n", rejected_text[:1200])
print("\nchosen_prompt_len =", sample["chosen_prompt_len"].item())
print("rejected_prompt_len =", sample["rejected_prompt_len"].item())

### 8.4.3 初始化 model 与 dataloader

In [ ]:
def init_dpo_experiment(cfg: Dict):
    wandb.init(
        project=cfg["wandb_project"],
        name=cfg["wandb_run_name"],
        config=cfg,
    )

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

    policy_model = AutoModelForCausalLM.from_pretrained(
        cfg["policy_model_path"],
        torch_dtype=dtype,
        attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
    ).to(cfg["policy_device"])

    ref_model = AutoModelForCausalLM.from_pretrained(
        cfg["reference_model_path"],
        torch_dtype=dtype,
        attn_implementation="flash_attention_2" if torch.cuda.is_available() else "eager",
    ).to(cfg["reference_device"])

    ref_model.eval()
    for p in ref_model.parameters():
        p.requires_grad = False

    return policy_model, ref_model

In [ ]:
def build_dpo_dataloaders(train_examples, val_examples, tokenizer, cfg: Dict):
    train_loader = DataLoader(
        HHPreferenceDataset(train_examples, tokenizer, cfg["max_length"]),
        batch_size=cfg["micro_batch_size"],
        shuffle=True,
    )

    val_loader = DataLoader(
        HHPreferenceDataset(val_examples, tokenizer, cfg["max_length"]),
        batch_size=cfg["micro_batch_size"],
        shuffle=False,
    )

    return train_loader, val_loader

In [ ]:
policy_model, ref_model = init_dpo_experiment(CFG)
train_loader, val_loader = build_dpo_dataloaders(train_examples, val_examples, tokenizer, CFG)

batch = next(iter(train_loader))
for k, v in batch.items():
    print(k, v.shape)

### 8.4.4 DPO 核心函数
依旧只计算 response 段的 log-prob，保证：
- prompt 不参与 chosen/rejected 的比较
- 只比较 response token 的对数概率和

In [ ]:
def compute_response_logprobs(
    logits: torch.Tensor,          # [B, L, V]
    input_ids: torch.Tensor,       # [B, L]
    attention_mask: torch.Tensor,  # [B, L]
    prompt_lengths: torch.Tensor,  # [B]
):
    """
    计算 response 段的总 log-prob。
    返回:
      seq_log_probs: [B]
      seq_lengths:   [B]
    """
    log_probs = F.log_softmax(logits, dim=-1)

    shift_log_probs = log_probs[:, :-1, :]
    shift_input_ids = input_ids[:, 1:]
    shift_attention_mask = attention_mask[:, 1:]

    token_log_probs = torch.gather(
        shift_log_probs,
        dim=2,
        index=shift_input_ids.unsqueeze(-1)
    ).squeeze(-1)

    response_mask = torch.zeros_like(shift_attention_mask)
    for i in range(token_log_probs.shape[0]):
        start = max(int(prompt_lengths[i].item()) - 1, 0)
        response_mask[i, start:] = 1

    combined_mask = shift_attention_mask * response_mask

    seq_log_probs = (token_log_probs * combined_mask).sum(dim=1)
    seq_lengths = combined_mask.sum(dim=1)

    return seq_log_probs, seq_lengths

In [ ]:
def compute_dpo_batch_loss(
    policy_model,
    ref_model,
    batch: Dict[str, torch.Tensor],
    beta: float,
    policy_device: str,
    reference_device: str,
):
    c_ids = batch["chosen_input_ids"].to(policy_device)
    r_ids = batch["rejected_input_ids"].to(policy_device)

    c_mask = batch["chosen_attention_mask"].to(policy_device)
    r_mask = batch["rejected_attention_mask"].to(policy_device)

    c_p_len = batch["chosen_prompt_len"].to(policy_device)
    r_p_len = batch["rejected_prompt_len"].to(policy_device)

    # policy logits
    c_logits = policy_model(c_ids, attention_mask=c_mask).logits
    r_logits = policy_model(r_ids, attention_mask=r_mask).logits

    # reference logits
    with torch.no_grad():
        c_logits_ref = ref_model(
            c_ids.to(reference_device),
            attention_mask=c_mask.to(reference_device)
        ).logits.to(policy_device)

        r_logits_ref = ref_model(
            r_ids.to(reference_device),
            attention_mask=r_mask.to(reference_device)
        ).logits.to(policy_device)

    # response log-probs
    c_lp, c_len = compute_response_logprobs(c_logits, c_ids, c_mask, c_p_len)
    c_lp_ref, _ = compute_response_logprobs(c_logits_ref, c_ids, c_mask, c_p_len)

    r_lp, r_len = compute_response_logprobs(r_logits, r_ids, r_mask, r_p_len)
    r_lp_ref, _ = compute_response_logprobs(r_logits_ref, r_ids, r_mask, r_p_len)

    # implicit rewards
    c_reward = beta * (c_lp - c_lp_ref)
    r_reward = beta * (r_lp - r_lp_ref)

    # DPO loss
    margin = c_reward - r_reward
    loss = -F.logsigmoid(margin).mean()

    metrics = {
        "loss": loss,
        "chosen_reward": c_reward.mean().item(),
        "rejected_reward": r_reward.mean().item(),
        "reward_margin": margin.mean().item(),
        "accuracy": (c_reward > r_reward).float().mean().item(),
        "chosen_logprob": c_lp.mean().item(),
        "rejected_logprob": r_lp.mean().item(),
        "chosen_len": c_len.float().mean().item(),
        "rejected_len": r_len.float().mean().item(),
    }

    return loss, metrics

先手算一个 batch，确认 loss 歪没歪：

In [ ]:
batch = next(iter(train_loader))

loss, metrics = compute_dpo_batch_loss(
    policy_model=policy_model,
    ref_model=ref_model,
    batch=batch,
    beta=CFG["beta"],
    policy_device=CFG["policy_device"],
    reference_device=CFG["reference_device"],
)

print("loss =", loss.item())
for k, v in metrics.items():
    if k != "loss":
        print(k, "=", v)

### 8.4.5 训练与验证循环

In [ ]:
@torch.no_grad()
def evaluate_dpo(
    policy_model,
    ref_model,
    loader,
    beta: float,
    policy_device: str,
    reference_device: str,
):
    policy_model.eval()

    losses = []
    accs = []
    margins = []

    for batch in loader:
        loss, metrics = compute_dpo_batch_loss(
            policy_model=policy_model,
            ref_model=ref_model,
            batch=batch,
            beta=beta,
            policy_device=policy_device,
            reference_device=reference_device,
        )

        losses.append(loss.item())
        accs.append(metrics["accuracy"])
        margins.append(metrics["reward_margin"])

    policy_model.train()

    return {
        "val/loss": float(np.mean(losses)) if losses else 0.0,
        "val/accuracy": float(np.mean(accs)) if accs else 0.0,
        "val/reward_margin": float(np.mean(margins)) if margins else 0.0,
    }

In [ ]:
def train_one_dpo_optimizer_step(
    policy_model,
    ref_model,
    optimizer,
    train_loader_iter,
    grad_accum_steps: int,
    beta: float,
    policy_device: str,
    reference_device: str,
):
    policy_model.train()
    optimizer.zero_grad(set_to_none=True)

    running = {
        "loss": 0.0,
        "chosen_reward": 0.0,
        "rejected_reward": 0.0,
        "reward_margin": 0.0,
        "accuracy": 0.0,
        "chosen_logprob": 0.0,
        "rejected_logprob": 0.0,
    }

    actual_micro_steps = 0

    for _ in range(grad_accum_steps):
        try:
            batch = next(train_loader_iter)
        except StopIteration:
            break

        loss, metrics = compute_dpo_batch_loss(
            policy_model=policy_model,
            ref_model=ref_model,
            batch=batch,
            beta=beta,
            policy_device=policy_device,
            reference_device=reference_device,
        )

        (loss / grad_accum_steps).backward()

        running["loss"] += loss.item()
        running["chosen_reward"] += metrics["chosen_reward"]
        running["rejected_reward"] += metrics["rejected_reward"]
        running["reward_margin"] += metrics["reward_margin"]
        running["accuracy"] += metrics["accuracy"]
        running["chosen_logprob"] += metrics["chosen_logprob"]
        running["rejected_logprob"] += metrics["rejected_logprob"]

        actual_micro_steps += 1

    if actual_micro_steps == 0:
        return None

    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), 1.0)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    for k in running:
        running[k] /= actual_micro_steps

    return {
        "train/loss": running["loss"],
        "train/chosen_reward": running["chosen_reward"],
        "train/rejected_reward": running["rejected_reward"],
        "train/reward_margin": running["reward_margin"],
        "train/accuracy": running["accuracy"],
        "train/chosen_logprob": running["chosen_logprob"],
        "train/rejected_logprob": running["rejected_logprob"],
        "train/lr": optimizer.param_groups[0]["lr"],
    }

In [ ]:
def run_dpo_training(cfg: Dict):
    os.makedirs(cfg["output_dir"], exist_ok=True)

    optimizer = torch.optim.AdamW(policy_model.parameters(), lr=cfg["lr"])
    # 可以改成 RMSprop
    # optimizer = torch.optim.RMSprop(policy_model.parameters(), lr=cfg["lr"])

    step = 0
    progress_bar = tqdm(desc="DPO Training", total=None)

    for epoch in range(cfg["epochs"]):
        train_loader_iter = iter(train_loader)

        while True:
            train_metrics = train_one_dpo_optimizer_step(
                policy_model=policy_model,
                ref_model=ref_model,
                optimizer=optimizer,
                train_loader_iter=train_loader_iter,
                grad_accum_steps=cfg["gradient_accumulation_steps"],
                beta=cfg["beta"],
                policy_device=cfg["policy_device"],
                reference_device=cfg["reference_device"],
            )

            if train_metrics is None:
                break

            step += 1

            wandb.log(train_metrics, step=step)

            progress_bar.set_postfix({
                "loss": f"{train_metrics['train/loss']:.4f}",
                "margin": f"{train_metrics['train/reward_margin']:.4f}",
                "acc": f"{train_metrics['train/accuracy']:.4f}",
            })
            progress_bar.update(1)

            if step % cfg["eval_interval"] == 0:
                val_metrics = evaluate_dpo(
                    policy_model=policy_model,
                    ref_model=ref_model,
                    loader=val_loader,
                    beta=cfg["beta"],
                    policy_device=cfg["policy_device"],
                    reference_device=cfg["reference_device"],
                )
                wandb.log(val_metrics, step=step)

            if step % cfg["save_interval"] == 0:
                save_dir = os.path.join(cfg["output_dir"], f"checkpoint-{step}")
                os.makedirs(save_dir, exist_ok=True)
                policy_model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)

    policy_model.save_pretrained(cfg["output_dir"])
    tokenizer.save_pretrained(cfg["output_dir"])
    wandb.finish()

    return policy_model

In [ ]:
trained_policy = run_dpo_training(CFG)

- 数据层：load_hh_dataset、HHPreferenceDataset
- 概率层：compute_response_logprobs
- 损失层：compute_dpo_batch_loss
- 训练层：train_one_dpo_optimizer_step、evaluate_dpo、run_dpo_training